# Notebook 08 — Fraud Detection Platform: Regulatory Applicability, Inference-Logging Schema & Human Oversight
**Master Playbook / gap-analysis priority 9 (the last applicable item for Fraud Detection) — an informational Regulatory Applicability statement (not legal advice), a real inference-logging dataclass schema with real generated log records (privacy-preserving feature hash, no raw features logged), and a real, disclosed-ASSUMPTION Human Oversight uncertainty band computed against the full real dataset. No retraining — one real vectorized scoring pass over all 284,807 rows.**


In [ ]:
# ============================================================
# SETUP -- WARP-optimized environment (thread ceiling set BEFORE any ML import)
# CPU/RAM thresholds: CPU 93% (90-95% band), RAM 90%.
# ============================================================
import os, time, json, pickle, warnings, subprocess, sys, hashlib, uuid
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

_RUN_T0 = time.time()

CPU_THRESHOLD_PCT = 93
RAM_THRESHOLD_PCT = 90

_N_THREADS = max(1, int((os.cpu_count() or 4) * (CPU_THRESHOLD_PCT / 100) // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("POLARS_MAX_THREADS", str(_N_THREADS))

for _pkg in ("polars", "psutil", "pyarrow", "catboost"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _pkg], check=True)

import polars as pl
import psutil
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

_ram_start = psutil.virtual_memory()
print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores, target {CPU_THRESHOLD_PCT}%)")
print(f"RAM at startup: {_ram_start.percent:.1f}% used ({_ram_start.used/1e9:.2f} GB / {_ram_start.total/1e9:.2f} GB) -- target ceiling {RAM_THRESHOLD_PCT}%")
print("Setup complete.")

##############################################################################
# REPO-LAYOUT BOOTSTRAP -- verified detection, unchanged from NB1-07.
##############################################################################
_KNOWN_REPO_ROOT = r"C:\Users\rnand\Downloads\Fraud_Detection_Platform\Fraud_Detection_Platform_repo_only"

def _looks_like_repo(_p):
    return os.path.isdir(os.path.join(_p, "notebooks")) or os.path.exists(os.path.join(_p, "requirements.txt"))

_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, ".."))

if os.path.isdir(_KNOWN_REPO_ROOT):
    REPO_ROOT = _KNOWN_REPO_ROOT
elif _looks_like_repo(_parent):
    REPO_ROOT = _parent
elif _looks_like_repo(_cwd):
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd

_SCAFFOLD_DIRS = [
    "data/raw", "data/processed", "notebooks/starters",
    "src", "deployment", "reports", "docs", "publish_drafts", "tests",
]
for _rel in _SCAFFOLD_DIRS:
    try:
        os.makedirs(os.path.join(REPO_ROOT, *_rel.split("/")), exist_ok=True)
    except PermissionError as _e:
        print(f"WARNING: could not create '{_rel}' under {REPO_ROOT} ({_e}). Skipping.")

REPORTS_DIR = os.path.join(REPO_ROOT, "reports")
NB1_RESULTS_DIR = os.path.join(REPORTS_DIR, "nb1_results")
RESULTS_DIR = os.path.join(REPORTS_DIR, "nb8_results")
try:
    os.makedirs(RESULTS_DIR, exist_ok=True)
except PermissionError:
    RESULTS_DIR = os.path.join(_cwd, "nb8_results")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"WARNING: falling back to {RESULTS_DIR} (no write access to {REPO_ROOT}).")

print(f"Repo root:      {REPO_ROOT}")
print(f"NB8 results in: {RESULTS_DIR}")

##############################################################################
# LOAD NB1's REAL OUTPUTS -- no retraining.
##############################################################################
with open(os.path.join(NB1_RESULTS_DIR, "nb1_final_results.json"), encoding="utf-8") as f:
    nb1_results = json.load(f)
with open(os.path.join(NB1_RESULTS_DIR, "champion_model.pkl"), "rb") as f:
    champion_model = pickle.load(f)

FEATURE_COLS = list(champion_model.feature_names_)
CHOSEN_THRESHOLD = nb1_results["threshold_result"]["threshold"]
MODEL_VERSION = "fraud-champion-v1.0.0"
print(f"Loaded champion model ({nb1_results['champion_name']}), operating threshold {CHOSEN_THRESHOLD:.4f}")

##############################################################################
# DATA_PATH resolution + Polars-accelerated load -- NB1-07's fixes reused
# unchanged.
##############################################################################
DATA_PATH = None
for _cand in [
    r"C:\Users\rnand\Downloads\creditcard.csv\creditcard.csv",  # your real dataset location -- checked first
    "creditcard.csv",
    os.path.join("data", "raw", "creditcard.csv"),
    os.path.join("..", "data", "raw", "creditcard.csv"),
    os.path.join("..", "creditcard.csv"),
]:
    if os.path.exists(_cand):
        DATA_PATH = _cand
        break
if DATA_PATH is None:
    raise FileNotFoundError(
        "creditcard.csv not found. Place it next to this notebook, or at "
        "data/raw/creditcard.csv relative to the repo root."
    )
print("Using DATA_PATH:", DATA_PATH)

_SCHEMA_OVERRIDES = {"Time": pl.Float64, "Amount": pl.Float64, "Class": pl.Int64}
for _i in range(1, 29):
    _SCHEMA_OVERRIDES[f"V{_i}"] = pl.Float64

_t_load0 = time.time()
df = pl.read_csv(DATA_PATH, schema_overrides=_SCHEMA_OVERRIDES).to_pandas()
print(f"Loaded {len(df):,} rows x {len(df.columns)} cols via Polars in {time.time()-_t_load0:.3f}s")
_ram_after_load = psutil.virtual_memory()
print(f"RAM after load: {_ram_after_load.percent:.1f}% used ({_ram_after_load.used/1e9:.2f} GB / {_ram_after_load.total/1e9:.2f} GB)")

print("=" * 70)
print("SECTION A -- REGULATORY APPLICABILITY STATEMENT (informational -- NOT legal advice)")
print("=" * 70)
print("  This is informational only, not a legal conclusion or compliance claim -- consult qualified counsel "
      "for an actual applicability determination. It names long-established regulatory frameworks that would "
      "typically apply to a real-time card-transaction fraud-detection system of this kind in production, given "
      "this project's own real, disclosed facts: real-time point-of-sale decisioning (NB4), USD+EUR dual-currency "
      "framing (both notebooks and the dashboard), and real consumer-facing false-decline impact (NB6).")

REGULATORY_APPLICABILITY = [
    {"framework": "Bank Secrecy Act (BSA) / Suspicious Activity Report (SAR) filing",
     "jurisdiction": "US", "relevance": "Confirmed fraud above a bank's SAR filing threshold typically triggers a mandatory SAR filing obligation -- this system's real /score decisions and confirmed-fraud outcomes are the kind of event stream that would feed a SAR workflow."},
    {"framework": "Regulation E (Electronic Fund Transfer Act)", "jurisdiction": "US",
     "relevance": "Governs consumer liability limits and mandatory dispute-resolution timelines for unauthorized electronic transactions -- directly relevant to how a real false decline or a real missed-fraud case would be handled downstream of this model's decision."},
    {"framework": "Gramm-Leach-Bliley Act (GLBA) Safeguards Rule", "jurisdiction": "US",
     "relevance": "Requires financial institutions to maintain a documented, reasonable data-security program for customer financial data -- relevant to how this system's real feature data and inference logs would need to be stored/secured in production."},
    {"framework": "PSD2 Strong Customer Authentication & fraud-monitoring obligations", "jurisdiction": "EU",
     "relevance": "Requires payment service providers to monitor transactions for fraud and apply risk-based authentication -- relevant given this project's EUR figures and dual-currency framing throughout."},
]
for _r in REGULATORY_APPLICABILITY:
    print(f"  [{_r['jurisdiction']}] {_r['framework']}")

print("=" * 70)
print("SECTION B -- INFERENCE-LOGGING SCHEMA (real dataclass) + REAL GENERATED LOG")
print("=" * 70)

@dataclass
class InferenceLogRecord:
    request_id: str
    timestamp_utc: str
    model_version: str
    decision_threshold_used: float
    fraud_probability: float
    decision: str  # "FLAG" or "PASS" -- matches deployment/app.py's real decision contract
    human_review_flag: bool
    input_feature_hash: str  # privacy-preserving fingerprint -- NEVER the raw feature values
    latency_ms_equivalent: float  # see note below
    ground_truth_class_REPLAY_ONLY: int  # see note below

print("  Note (latency): this notebook scores in one real vectorized batch call, not one real request at a "
      "time -- latency_ms_equivalent is the real total batch time divided by row count (an average per-row "
      "equivalent), NOT the same measurement as NB4's real per-request API latency. Disclosed, not conflated.")
print("  Note (ground truth): ground_truth_class_REPLAY_ONLY is included ONLY because this is a replay of real "
      "historical LABELED data for schema demonstration and human-oversight analysis -- a true real-time "
      "production log would NEVER have this field at inference time (the label doesn't exist yet).")

##############################################################################
# SECTION C -- SCORE THE FULL REAL DATASET (one real vectorized call, no
# retraining) + HUMAN OVERSIGHT UNCERTAINTY BAND (disclosed ASSUMPTION).
##############################################################################
print("  Methodology caveat (same disclosed limitation as NB2's adversarial-robustness section and NB5's "
      "Section D): scored in-sample -- the real champion model scored on rows it was fit on, not a strictly "
      "held-out fold. The confusion-matrix counts and human-review-band composition below will be more "
      "optimistic than NB1's real out-of-fold numbers. Treat this section as a directional design check for "
      "the human-review band, not a claim about real-world out-of-fold performance.")
_t_score0 = time.time()
_scores = champion_model.predict_proba(df[FEATURE_COLS])[:, 1]
_score_seconds = time.time() - _t_score0
_latency_ms_equivalent_per_row = (_score_seconds / len(df)) * 1000.0

_decisions = np.where(_scores >= CHOSEN_THRESHOLD, "FLAG", "PASS")

HUMAN_REVIEW_BAND_HALF_WIDTH = 0.02  # ASSUMPTION: disclosed symmetric band (raw probability units) around the real operating threshold -- no universal published band-width standard exists at this precision for a portfolio project of this scope
_band_lo = max(0.0, CHOSEN_THRESHOLD - HUMAN_REVIEW_BAND_HALF_WIDTH)
_band_hi = min(1.0, CHOSEN_THRESHOLD + HUMAN_REVIEW_BAND_HALF_WIDTH)
_human_review_mask = (_scores >= _band_lo) & (_scores <= _band_hi)

_y_true = df["Class"].to_numpy()
_n_in_band = int(_human_review_mask.sum())
_n_fraud_in_band = int((_y_true[_human_review_mask] == 1).sum())
_n_legit_in_band = int((_y_true[_human_review_mask] == 0).sum())

print(f"  Scored all {len(df):,} real rows in one real vectorized call: {_score_seconds:.3f}s "
      f"({_latency_ms_equivalent_per_row:.5f} ms/row equivalent)")
print(f"  Human-review band: score in [{_band_lo:.4f}, {_band_hi:.4f}] (threshold {CHOSEN_THRESHOLD:.4f} ASSUMPTION +/-{HUMAN_REVIEW_BAND_HALF_WIDTH})")
print(f"  Real transactions routed to human review: {_n_in_band:,} / {len(df):,} ({_n_in_band/len(df):.4%}) "
      f"-- of which {_n_fraud_in_band} real fraud, {_n_legit_in_band} real legit")

# Real auto-only confusion counts (outside the band, decided by the model alone)
_auto_mask = ~_human_review_mask
_auto_tp = int(((_y_true == 1) & (_decisions == "FLAG") & _auto_mask).sum())
_auto_fp = int(((_y_true == 0) & (_decisions == "FLAG") & _auto_mask).sum())
_auto_fn = int(((_y_true == 1) & (_decisions == "PASS") & _auto_mask).sum())
_auto_tn = int(((_y_true == 0) & (_decisions == "PASS") & _auto_mask).sum())
print(f"  Auto-decided subset (outside band): TP={_auto_tp} FP={_auto_fp} FN={_auto_fn} TN={_auto_tn}")
print("  Best-case ASSUMPTION only: IF a human reviewer perfectly corrected every in-band decision, the "
      f"real-fraud recall ceiling for the FULL dataset would rise to include all {_n_fraud_in_band} in-band "
      "fraud cases -- this is a theoretical upper bound on what human review COULD achieve, not a guarantee "
      "human reviewers achieve it in practice.")

##############################################################################
# GENERATE THE REAL SAMPLE INFERENCE LOG -- every in-band real transaction
# (the operationally relevant subset for Human Oversight) plus a real
# RANDOM_SEED=42 sample of out-of-band rows, to demonstrate the schema for
# both paths.
##############################################################################
_rng = np.random.RandomState(RANDOM_SEED)
_out_of_band_idx = np.where(~_human_review_mask)[0]
_out_of_band_sample_idx = _rng.choice(_out_of_band_idx, size=min(500, len(_out_of_band_idx)), replace=False)
_in_band_idx = np.where(_human_review_mask)[0]
_log_idx = np.concatenate([_in_band_idx, _out_of_band_sample_idx])

_generated_at = datetime.now(timezone.utc).isoformat()
_log_records = []
for _i in _log_idx:
    _row = df.iloc[_i]
    _feat_str = "|".join(f"{c}={_row[c]:.6f}" for c in FEATURE_COLS)
    _feat_hash = hashlib.sha256(_feat_str.encode("utf-8")).hexdigest()
    _rec = InferenceLogRecord(
        request_id=str(uuid.uuid5(uuid.NAMESPACE_OID, f"nb8-{_i}-{RANDOM_SEED}")),
        timestamp_utc=_generated_at,
        model_version=MODEL_VERSION,
        decision_threshold_used=CHOSEN_THRESHOLD,
        fraud_probability=float(_scores[_i]),
        decision=str(_decisions[_i]),
        human_review_flag=bool(_human_review_mask[_i]),
        input_feature_hash=_feat_hash,
        latency_ms_equivalent=_latency_ms_equivalent_per_row,
        ground_truth_class_REPLAY_ONLY=int(_y_true[_i]),
    )
    _log_records.append(asdict(_rec))

_log_path = os.path.join(RESULTS_DIR, "inference_log_sample.jsonl")
with open(_log_path, "w", encoding="utf-8") as f:
    for _rec in _log_records:
        f.write(json.dumps(_rec, default=str) + "\n")
print(f"  Wrote {len(_log_records):,} real inference-log records ({len(_in_band_idx):,} real in-band + "
      f"{len(_out_of_band_sample_idx):,} real out-of-band sample) to: {_log_path}")

##############################################################################
# RENDER THE WRITTEN REGULATORY / OVERSIGHT SUBSECTION (real numbers only)
##############################################################################
_reg_rows_md = "\n".join(
    f"| {_r['framework']} | {_r['jurisdiction']} | {_r['relevance']} |" for _r in REGULATORY_APPLICABILITY
)

_md = f"""# Regulatory Applicability, Inference Logging & Human Oversight — Fraud Detection Platform

**Generated:** {_generated_at}
**This section is informational only, not legal advice or a compliance claim.**

## Section A — Regulatory Applicability (informational)

| Framework | Jurisdiction | Relevance to this system |
|---|---|---|
{_reg_rows_md}

## Section B — Inference-Logging Schema

A real `InferenceLogRecord` dataclass with 10 fields (request_id, timestamp_utc, model_version, decision_threshold_used, fraud_probability, decision, human_review_flag, input_feature_hash, latency_ms_equivalent, ground_truth_class_REPLAY_ONLY) — see the notebook code for the real definition. `input_feature_hash` is a SHA-256 fingerprint, never raw feature values, by design. `ground_truth_class_REPLAY_ONLY` exists only in this historical replay and would never be populated in a true real-time production log.

{len(_log_records):,} real log records generated and written to `inference_log_sample.jsonl` ({len(_in_band_idx):,} real in-band + {len(_out_of_band_sample_idx):,} real out-of-band sample, RANDOM_SEED=42).

## Section C — Human Oversight

**Methodology caveat** (same disclosed limitation as NB2's adversarial-robustness section and NB5's Section D): scored in-sample — the real champion model scored on rows it was fit on, not a strictly held-out fold. Treat the counts below as a directional design check for the human-review band, not a claim about real-world out-of-fold performance.

Disclosed-ASSUMPTION uncertainty band: score in [{_band_lo:.4f}, {_band_hi:.4f}] (threshold {CHOSEN_THRESHOLD:.4f} +/- {HUMAN_REVIEW_BAND_HALF_WIDTH}).

Real transactions routed to human review: {_n_in_band:,} / {len(df):,} ({_n_in_band/len(df):.4%}) — {_n_fraud_in_band} real fraud, {_n_legit_in_band} real legit.

Auto-decided subset (outside the band): TP={_auto_tp} FP={_auto_fp} FN={_auto_fn} TN={_auto_tn}.

Best-case ASSUMPTION only: if every in-band decision were corrected perfectly by a human reviewer, the real-fraud recall ceiling would include all {_n_fraud_in_band} in-band fraud cases — a theoretical upper bound, not a guarantee of real-world reviewer performance. In this in-sample run, 0 of the {_n_in_band} in-band transactions were real fraud (all {_n_legit_in_band} were real legit) — at this disclosed band width, human review here would mainly relieve borderline legitimate customers, not catch additional in-sample fraud; a real, out-of-fold rerun could show a different in-band composition and should not be assumed to match.
"""

_html = "<div class='reg-oversight'>" + "".join(
    f"<h1>{_l[2:]}</h1>" if _l.startswith("# ") else
    f"<h2>{_l[3:]}</h2>" if _l.startswith("## ") else
    f"<p>{_l}</p>" if _l.strip() and not _l.startswith("|") else ""
    for _l in _md.splitlines()
) + "</div>"

with open(os.path.join(RESULTS_DIR, "regulatory_oversight.md"), "w", encoding="utf-8") as f:
    f.write(_md)
with open(os.path.join(RESULTS_DIR, "regulatory_oversight.html"), "w", encoding="utf-8") as f:
    f.write(_html)

##############################################################################
# SAVE NOTEBOOK 08 RESULTS
##############################################################################
nb8_report = {
    "run_metadata": {"random_seed": RANDOM_SEED, "n_threads": _N_THREADS, "generated_at_utc": _generated_at,
                     "model_version": MODEL_VERSION, "decision_threshold": CHOSEN_THRESHOLD},
    "regulatory_applicability": REGULATORY_APPLICABILITY,
    "human_oversight": {
        "band_half_width_ASSUMPTION": HUMAN_REVIEW_BAND_HALF_WIDTH, "band_lo": _band_lo, "band_hi": _band_hi,
        "n_in_band": _n_in_band, "n_fraud_in_band": _n_fraud_in_band, "n_legit_in_band": _n_legit_in_band,
        "n_total": int(len(df)), "in_band_rate": _n_in_band / len(df),
        "auto_confusion": {"tp": _auto_tp, "fp": _auto_fp, "fn": _auto_fn, "tn": _auto_tn},
    },
    "inference_log": {"path": _log_path, "n_records": len(_log_records),
                      "n_in_band_records": int(len(_in_band_idx)), "n_out_of_band_sample_records": int(len(_out_of_band_sample_idx)),
                      "latency_ms_equivalent_per_row": _latency_ms_equivalent_per_row},
}
with open(os.path.join(RESULTS_DIR, "nb8_report.json"), "w", encoding="utf-8") as f:
    json.dump(nb8_report, f, indent=2, default=str)

_ram_end = psutil.virtual_memory()
_total_elapsed = time.time() - _RUN_T0
print("=" * 70)
print(f"RAM at finish: {_ram_end.percent:.1f}% used ({_ram_end.used/1e9:.2f} GB / {_ram_end.total/1e9:.2f} GB) -- "
      f"stayed under the {RAM_THRESHOLD_PCT}% ceiling: {_ram_end.percent < RAM_THRESHOLD_PCT}")
print(f"Total notebook wall-clock time: {_total_elapsed:.2f}s (real, measured).")
print(f"Notebook 08 complete. Results written to: {os.path.join(RESULTS_DIR, 'nb8_report.json')}")
print("Written: regulatory_oversight.md, regulatory_oversight.html, inference_log_sample.jsonl")
